<a href="https://colab.research.google.com/github/StillWork/6FM2/blob/main/FM2_04_%E1%84%82%E1%85%B2%E1%84%89%E1%85%B3%E1%84%80%E1%85%A5%E1%86%B7%E1%84%89%E1%85%A2%E1%86%A8_%E1%84%8B%E1%85%A6%E1%84%8B%E1%85%B5%E1%84%8C%E1%85%A5%E1%86%AB%E1%84%90%E1%85%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 claude.ai 프롬프트 / claude.ai Prompt

> **수업 활용법:** 아래 프롬프트를 [claude.ai](https://claude.ai)에 붙여넣으면 유사한 노트북을 생성할 수 있습니다.

---

### 🇰🇷 한국어 프롬프트
```
Anthropic Claude API + feedparser(Google 뉴스 RSS)를 이용한
뉴스 검색 AI 에이전트 Google Colab 노트북을 한국어로 작성해주세요.

1. 개념: '외부 데이터 연동 에이전트' — 에이전트가 실시간 외부 데이터를
   가져와 LLM의 지식 한계를 극복하는 방법 설명
2. 도구: search_news(query, days=7)
   - feedparser로 Google 뉴스 RSS 검색
   - 최근 N일 이내 기사만 필터링
   - 제목·날짜·출처·링크 반환
3. 에이전트 루프 구현 (FM2-02와 동일한 패턴)
4. 에이전트 역할: 검색 결과에서 가장 중요한 뉴스 10개를 선정하고
   번호·제목·날짜·출처·링크를 표로 출력,
   10개 뉴스를 50단어 이내로 종합 요약
5. 미니 실습: 사용자가 직접 검색어를 입력(input())하여 뉴스 검색

설치: anthropic, feedparser. 한국어 주석. 초보자 친화적.
```

### 🇺🇸 English Prompt
```
Create a Korean Google Colab notebook for a News Search AI Agent
using Anthropic Claude API + feedparser (Google News RSS).

1. Concept: 'External Data Agent' — how agents overcome LLM knowledge cutoffs
2. Tool: search_news(query, days=7)
   - Fetch Google News RSS with feedparser
   - Filter articles from last N days
   - Return title, date, source, link
3. Implement agent loop (same pattern as FM2-02)
4. Agent behavior: from search results, select top 10 most important news,
   display as numbered list with title/date/source/link,
   summarize all 10 in ~50 words
5. Mini exercise: user types their own query via input()

Install: anthropic, feedparser. Korean comments. Beginner-friendly.
```

# FM2 실습 4: 뉴스 검색 AI 에이전트

## 학습 목표
- 에이전트가 실시간 외부 데이터를 가져오는 방법을 구현한다
- Google 뉴스 RSS를 도구로 연동하는 방법을 배운다
- Claude가 검색 결과를 분석·정리하는 과정을 이해한다

## 이 실습의 핵심 개념: 외부 데이터 연동 에이전트

| 한계 | 해결책 |
|------|--------|
| Claude의 학습 데이터는 특정 날짜에서 끊김 | 도구로 실시간 뉴스를 가져옴 |
| LLM은 최신 정보를 모름 | 에이전트가 인터넷에서 검색 후 Claude에 전달 |
| 환각(없는 정보 생성) 위험 | 실제 기사 URL을 출처로 제공 |

```
[에이전트 흐름]
"삼성전자 최근 뉴스 알려줘"
    ↓
Claude → search_news("삼성전자", days=7) 호출
    ↓
Google 뉴스 RSS → 기사 목록 반환
    ↓
Claude → 중요도 판단 → 상위 10개 선정 + 요약
```

In [2]:
!pip install anthropic feedparser -q
import getpass, anthropic
import feedparser
import urllib.parse
from datetime import datetime, timezone, timedelta

api_key = getpass.getpass("🔑 Anthropic API 키: ")
client = anthropic.Anthropic(api_key=api_key)
print("✅ 준비 완료!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.7 MB/s eta 0:00:00
🔑 Anthropic API 키: ··········
✅ 준비 완료!


## 1단계: 뉴스 검색 도구 정의

Google 뉴스는 **RSS 피드**를 무료로 제공합니다.  
`feedparser` 라이브러리로 RSS를 읽으면 별도 API 키 없이 뉴스를 가져올 수 있습니다.

In [3]:
def search_news(query: str, days: int = 7) -> str:
    """
    Google 뉴스 RSS로 최근 기사를 검색합니다.

    Args:
        query: 검색할 키워드 또는 회사명
        days:  검색 기간 (최근 N일, 기본값 7)

    Returns:
        기사 목록 문자열 (제목·날짜·출처·링크)
    """
    encoded = urllib.parse.quote(query)
    url = f"https://news.google.com/rss/search?q={encoded}&hl=ko&gl=KR&ceid=KR:ko"

    feed = feedparser.parse(url)

    if not feed.entries:
        return f"'{query}'에 대한 뉴스를 찾을 수 없습니다."

    # 속도를 위해 최대 25개 항목만 확인
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)
    articles = []

    for entry in feed.entries[:25]:
        try:
            pub = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
        except Exception:
            continue

        if pub < cutoff:
            continue

        source = getattr(getattr(entry, 'source', None), 'title', '출처 불명')
        articles.append({
            'title':  entry.title,
            'date':   pub.strftime('%Y-%m-%d'),
            'source': source,
            'link':   entry.link,
        })

    if not articles:
        return f"최근 {days}일 내 '{query}' 관련 기사가 없습니다."

    # 프롬프트 크기를 줄이기 위해 최대 15개로 제한
    articles = articles[:15]

    lines = [f"🔍 '{query}' 검색 결과 — 최근 {days}일, {len(articles)}건\n"]
    for i, a in enumerate(articles, 1):
        lines.append(f"{i}. [{a['date']}] {a['title']}")
        lines.append(f"   출처: {a['source']}")
        lines.append(f"   링크: {a['link']}")
        lines.append("")

    return "\n".join(lines)


# 도구 동작 확인
print("도구 테스트 중...")
test_result = search_news("인공지능", days=7)
print(test_result[:500], "...")

도구 테스트 중...
🔍 '인공지능' 검색 결과 — 최근 7일, 15건

1. [2026-03-18] 국방 AI 경쟁력, 반도체 주권에 달렸다…국산 NPU·생태계 구축 시급 - 지디넷코리아
   출처: 지디넷코리아
   링크: https://news.google.com/rss/articles/CBMiVkFVX3lxTE9GdXB1UXFkYVNBS2trNGoyb1dGcjlBX0w4Y2RXUG1CZ0dZdmg0NkYwazBQaWpKV2FNSFk1V0JmYzNjUk8tNnRZdURhNE1NdHpiSVQwZVRB?oc=5

2. [2026-03-18] 배경훈 “AI, 독자성 논란 떠나 세계적 수준 못 만들면 무의미…2~3년이 승부처” - 한겨레
   출처: 한겨레
   링크: https://news.google.com/rss/articles/CBMiYEFVX3lxTE14bks0MDJuR0RjV0tKS0lrNEd1YU96M0ZLVmtMb1N5dkx1VS1sS3pzWjB6c2VnU3ZlTlQ3ZFhSRXJ1TGlLY ...


In [4]:
# ─── 도구 스키마 + 매핑 ────────────────────────────────────────
TOOLS = [
    {
        "name": "search_news",
        "description": (
            "Google 뉴스에서 특정 키워드나 회사명으로 최근 뉴스를 검색합니다. "
            "최신 정보나 특정 회사/주제 관련 뉴스가 필요할 때 사용하세요."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "검색할 키워드 또는 회사명 (예: '삼성전자', 'AI 반도체')"
                },
                "days": {
                    "type": "integer",
                    "description": "검색 기간 (일 수, 기본값 7)"
                }
            },
            "required": ["query"]
        }
    }
]

TOOL_FUNCTIONS = {"search_news": search_news}

print("✅ 도구 스키마 준비 완료!")

✅ 도구 스키마 준비 완료!


## 2단계: 뉴스 에이전트 루프

In [5]:
from IPython.display import display, Markdown

NEWS_SYSTEM_PROMPT = """당신은 뉴스 분석 전문 에이전트입니다.

역할:
1. search_news 도구로 관련 뉴스를 검색합니다
2. 검색 결과에서 가장 중요한 뉴스 최대 10개를 선정합니다
3. 반드시 아래 형식으로 출력합니다:

## 📰 [검색어] 주요 뉴스 TOP 10 (최근 7일)

| # | 제목 | 날짜 | 출처 | 링크 |
|---|------|------|------|------|
| 1 | 제목 | YYYY-MM-DD | 출처명 | [링크](URL) |

(링크는 반드시 마크다운 형식 [링크](실제URL) 으로 작성하고, URL을 직접 노출하지 마세요)

## 📝 종합 요약 (50단어 이내)
(10개 뉴스 전체를 아우르는 핵심 트렌드를 50단어 이내로 반드시 작성하세요)

중요도 판단 기준: 업계 파급력, 사회적 관심도, 새로운 정보 여부"""


def run_news_agent(user_query, verbose=True):
    """뉴스 검색 에이전트를 실행합니다."""
    messages = [{"role": "user", "content": user_query}]

    if verbose:
        print(f"\n❓ 요청: {user_query}")
        print("─" * 65)

    MAX_TURNS = 5  # 무한 루프 방지

    for turn in range(MAX_TURNS):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=3000,          # 표 10행 + 요약이 모두 들어갈 충분한 크기
            system=NEWS_SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        # ── 케이스 1: 최종 답변 또는 토큰 초과 ─────────────────────
        # ⚠️ "max_tokens"도 end_turn과 동일하게 처리해야 합니다.
        #    그렇지 않으면 assistant 메시지가 마지막인 채로 API가 재호출되어
        #    BadRequestError: "assistant message prefill" 오류가 발생합니다.
        if response.stop_reason in ("end_turn", "max_tokens"):
            final = next((b.text for b in response.content if hasattr(b, 'text')), "")
            # ✅ Markdown으로 렌더링 → [링크](URL) 형식이 클릭 가능한 하이퍼링크로 표시됨
            display(Markdown(final))
            return final

        # ── 케이스 2: 도구 사용 요청 ────────────────────────────────
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                tool_input = block.input  # ✅ block.input 사용

                if verbose:
                    print(f"  🔧 뉴스 검색 중: {tool_input}")

                result = TOOL_FUNCTIONS[block.name](**tool_input)

                if verbose:
                    print(f"  📦 {result.split(chr(10))[0]}")

                tool_results.append({
                    "type":        "tool_result",
                    "tool_use_id": block.id,
                    "content":     result
                })

            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            else:
                break  # tool_use 블록이 없는 예외 상황

        else:
            break  # 예상치 못한 stop_reason → 루프 종료

    return ""


print("✅ 뉴스 에이전트 준비 완료!")

✅ 뉴스 에이전트 준비 완료!


## 3단계: 테스트

## ✏️ 직접 검색해보기

궁금한 회사명이나 키워드를 입력하면 뉴스를 검색합니다.

In [6]:
print("🔍 뉴스 검색 에이전트")
user_keyword = input("검색할 회사명 또는 키워드를 입력하세요: ").strip()

if user_keyword:
    run_news_agent(f"{user_keyword} 관련 최근 1주일 주요 뉴스 TOP 10과 종합 요약을 보여줘")
else:
    print("검색어를 입력해주세요.")

🔍 뉴스 검색 에이전트
검색할 회사명 또는 키워드를 입력하세요: 한국제약바이오협회

❓ 요청: 한국제약바이오협회 관련 최근 1주일 주요 뉴스 TOP 10과 종합 요약을 보여줘
─────────────────────────────────────────────────────────────────
  🔧 뉴스 검색 중: {'query': '한국제약바이오협회', 'days': 7}
  📦 🔍 '한국제약바이오협회' 검색 결과 — 최근 7일, 10건


검색 결과를 바탕으로 정리해드립니다!

---

## 📰 한국제약바이오협회 주요 뉴스 TOP 10 (최근 7일)

| # | 제목 | 날짜 | 출처 | 링크 |
|---|------|------|------|------|
| 1 | 제약바이오협회 17대 이사장 '동국제약 권기범 회장' 선출 | 2026-03-15 | MEDI:GATE NEWS | [링크](https://news.google.com/rss/articles/CBMibEFVX3lxTE1KTElmMzNjaEN0bFhReUlRdWZqUzVwdk1aSjhkbWhzYUxCY3F1THJ3M3p6X3BnX05ZUmFGSTFfeFJEb080UmlUQ0h3MFpCZUJmS00xcEFON0ZFbTVrRzZPYU5feENlZDhaLUQ3SA?oc=5) |
| 2 | 정부, 약값 인하 충격 줄인다…10년 걸친 연착륙 추진 | 2026-03-16 | 세무사신문 | [링크](https://news.google.com/rss/articles/CBMibkFVX3lxTE1tQWkyVnBfM2tuaC1rbVRtMVFoZ3ZUQjUzdVg2bVo3S3dFSWowdmlKUWxvb0NHS3d2eFZ5bEdEWUZwSUZxMHVqdjZkMUEyM2xUZjVVejVuT1FIU2JCSmg1ckpUNE5FeHFMRUFPZEp3?oc=5) |
| 3 | 바이오 차이나에서 주목받은 K-바이오…수출 계약서만 남았다 | 2026-03-16 | 메디칼타임즈 | [링크](https://news.google.com/rss/articles/CBMicEFVX3lxTE1sU3NZSzhjYlg2M0xqYnkyZ2lLRl9Uckx2TnZ0b3dSQ0tWWDFBWTV5Y1c0V1p4N3BRNG4xVC1ZM0RnSW05ak0xZFJtQVNnaGlEckJNaEtKWmFSUE1wRXQ5Z0xBX0FaWEVMbFl4WjFwakQ?oc=5) |
| 4 | 유가·환율 급등에…'제약·바이오' 부담 커진다 | 2026-03-19 | 네이트 | [링크](https://news.google.com/rss/articles/CBMiU0FVX3lxTE9WY0dMWFAxQWgtWFNXT2N0NmIyaExRMXRqTU1tVlNpUmlvOVRlZi1LRUZOS1VNdzF4eGdqT3VUTkpuVGNSRlRna3pUQ19SdHF6QW5Z?oc=5) |
| 5 | '양적 성장' 이룬 K제약바이오…다음 과제는 '질적 경쟁력' | 2026-03-15 | EBN | [링크](https://news.google.com/rss/articles/CBMiaEFVX3lxTE8ybGgybUJpNnVwNGlWOWhqaFRBOHA0WGIzZ0wzTFMzdUxnVl9JbmdwejdkZ1UzRlA0WDgxcm1oMllUV3RfN0RTNWtRWnd6NnNtVDV6akJsMzdKaFZ3M0E4VFY2SzdSTHBx?oc=5) |
| 6 | 약값 아끼면 보상 확대…건보 재정 절감 위한 '시장연동형' 제도 도입 | 2026-03-18 | 더쎈뉴스 | [링크](https://news.google.com/rss/articles/CBMiaEFVX3lxTE5oYW5DTGJ1RkJoUGhRUnU2LUtEeEY5NzJlTFBUaXZLeHhtZGJGNGRGdW5UVExuYmFMbVd3ZklLS2twbWhkazdIaWdLWXFqU1gxZHBERzNQd0NCUGsybUhrQmMzc3VtS0Vt?oc=5) |
| 7 | 약값 낮춘 민간 병원과 약국에 절감액 35% 돌려준다 | 2026-03-18 | 네이트 | [링크](https://news.google.com/rss/articles/CBMiU0FVX3lxTFBzRVI1TWwwWU4yWVU2eVdvRjJFeGFhVUgxellIMTNzSjlFSnpmRVV3dk1RVkZrYy04TFpoSkV2bDZNR215WXdiMzFJSTJudVdHZ1ln?oc=5) |
| 8 | 약값 낮춘 민간 병원과 약국에 절감액 35% 돌려주기로 (후속 보도) | 2026-03-18 | 네이트 | [링크](https://news.google.com/rss/articles/CBMiU0FVX3lxTE9EZjJQUFFIOERRYlNZd0gyRF84QWhxdV9yR0ZzbHZERm85TF9VZ2J5N2lZMzh5OVBibHFWTnQwRVoydUlNU1NCaFZReVdwbXNpaS04?oc=5) |
| 9 | 약값 거품 걷어내고 연구하는 강소기업에 성장 사다리 놓는다 | 2026-03-15 | 연합뉴스 | [링크](https://news.google.com/rss/articles/CBMiW0FVX3lxTE5QWmk5VGhIellVUW9nY0ZtdWFaREczWlh5bm9YRFRIRVNZSkFsaDRmak1aeWstcGVoOWtxQUdJNmJkamVfMWtDald4VE5aUHRnelgtamZWb014QWvSAWBBVV95cUxQQ25MR2JrSVYyRFhUUloxSjhKenBhbjZqMXh3TDVYVWE1R2VQVkhBczgybkRIQk1ZZnhyUzdSRWxEbTAxUUs0S0k2RTNEOTFkcVpIYVR1a3pzMXA3ZjZSTE0?oc=5) |
| 10 | 제10회 뉴시스 제약-바이오 포럼 기념촬영 | 2026-03-15 | 뉴시스 | [링크](https://news.google.com/rss/articles/CBMiYEFVX3lxTE5KNlhpTkpESko2cnlXNGhGbmRCT1NCM2U4RGlkTUg2LW5uMG13SHNFR0J6dnU5WjBQU3lmclhwY0FIM0dTQkp5UHlvSkY4VVNMZEZ6UlZUMWZtal9iNTJ4cA?oc=5) |

---

## 📝 종합 요약 (50단어 이내)

동국제약 권기범 회장이 한국제약바이오협회 17대 이사장으로 선출되며 리더십이 교체됐다. 정부는 약가 인하 연착륙과 시장연동형 절감 보상제를 추진하고, K-바이오는 중국 시장 진출과 질적 경쟁력 강화라는 과제를 안고 있다. 유가·환율 급등은 업계 부담을 가중시키고 있다.

## 📝 정리

| 요소 | 이번 실습 | 역할 |
|------|----------|------|
| 도구 | `search_news()` | 실시간 외부 데이터 수집 |
| 데이터 소스 | Google 뉴스 RSS | 최신 뉴스 (무료, API 키 불필요) |
| 에이전트 역할 | 중요도 판단 + 요약 | LLM의 분석·정리 능력 활용 |

**에이전트가 해결하는 LLM의 한계:**
- ❌ LLM 단독: "삼성전자 최근 뉴스" → 학습 시점 이후 정보 없음
- ✅ 에이전트: search_news 도구 → 오늘 뉴스 → 요약 제공

**실제 적용 아이디어:**
- 경쟁사 모니터링 자동화
- 투자 관심 종목 뉴스 요약
- 업계 트렌드 주간 브리핑 자동 생성

**다음 실습 (FM2-05):** Wikipedia 리서치 에이전트 — 노트 저장 도구를 추가하여 더 정교한 다단계 조사를 구현합니다!